# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/latent-play/blob/refactor/dynamics-lam-overhaul/notebooks/colab_training_and_demo.ipynb)

This notebook executes the end-to-end pipeline of **`latent-play`** on Google Colab Pro (**A100 GPU + High-RAM**) with **automatic checkpoint persistence to Google Drive every 5 minutes**:
- If Colab disconnects at any point, clicking **Run all** will automatically resume training from the latest checkpoint saved in your Google Drive (`MyDrive/latent_play_overhaul/`).
- Datasets (`data/`) are stored on Colab's local NVMe disk for maximum read throughput (`np.memmap`), while `checkpoints/` and `outputs/` are symlinked directly to your Google Drive.

## 1. Setup, Google Drive Mounting, and Repository Cloning

In [ ]:
import os, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# 1. Cleanly clone repository strictly on refactor/dynamics-lam-overhaul
import os
BRANCH = "refactor/dynamics-lam-overhaul"
!rm -rf /content/latent-play
!git clone -b {BRANCH} https://github.com/rivas-vega/latent-play.git /content/latent-play
%cd /content/latent-play
!git branch -v
os.environ["PYTHONPATH"] = "/content/latent-play"

# 2. Mount Google Drive and symlink to a clean isolated directory
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/latent_play_overhaul"
    os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
    for folder in ("checkpoints", "outputs"):
        if os.path.islink(folder):
            os.unlink(folder)
        elif os.path.isdir(folder):
            !rm -rf {folder}
        os.symlink(f"{DRIVE_ROOT}/{folder}", folder)
    print(f"Checkpoints and outputs symlinked to Google Drive: {DRIVE_ROOT}")
except Exception as e:
    print(f"Running without Google Drive ({e}): saving to local disk.")

# 3. Install dependencies and run the 47 unit tests
!pip install -q -e ".[crafter,dev]" matplotlib
!pytest -q


## 2. Crafter Experiment: Quick Validation vs. Full Benchmark
Crafter is the primary open-world survival benchmark for `latent-play`.
Select `fast_validation` for a ~12 min end-to-end smoke test, or `full_benchmark` for the final 30K-step paper run.

In [ ]:
# @title Select Crafter Experiment Scale
# @markdown Choose between a fast 12-minute smoke test or the full 30k-step benchmark.
CRAFTER_MODE = "fast_validation"  # @param ["fast_validation", "full_benchmark"]

if CRAFTER_MODE == "fast_validation":
    CRAFTER_STEPS = 50000
    TRAIN_STEPS = 2500
    WARMUP_STEPS = 250
    EVAL_EVERY = 500
    SWEEP_EPISODES = 10
    SWEEP_SEEDS = "0"
    SWEEP_FRACTIONS = "0.01 0.05 0.1 1.0"
else:
    CRAFTER_STEPS = 300000
    TRAIN_STEPS = 30000
    WARMUP_STEPS = 1000
    EVAL_EVERY = 2000
    SWEEP_EPISODES = 30
    SWEEP_SEEDS = "0 1 2"
    SWEEP_FRACTIONS = "0.001 0.01 0.05 0.1 1.0"

print(f"Active Crafter Mode: {CRAFTER_MODE}")
print(f"Data steps: {CRAFTER_STEPS} | Model train steps: {TRAIN_STEPS} | Eval episodes: {SWEEP_EPISODES}")


In [ ]:
!if [ ! -f data/crafter/meta.json ]; then python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps {CRAFTER_STEPS} --workers 4; fi
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif


## 3. Train Latent Action Model (LAM) on `Crafter` (`label_fraction=0.0`)
Automatically saves checkpoints every 5 minutes to your Google Drive (`MyDrive/latent_play_overhaul/checkpoints/lam_crafter`).

In [ ]:
!python scripts/train_lam.py --config configs/lam_crafter.yaml --force train.compile=true train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}


## 4. Train JEPA Dynamics + SIGReg on `Crafter` (Latent Actions vs. Oracle)
Automatically saves checkpoints every 5 minutes to your Google Drive (`dyn_crafter_lam` and `dyn_crafter_oracle`).

In [ ]:
# 5a. Our model (latent-play): conditioned purely on discovered LAM latent codes
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml --force lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.out_dir=checkpoints/dyn_crafter_lam train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}

# 5b. Action Oracle (Upper Bound): conditioned on 100% ground-truth Crafter actions
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml --force train.out_dir=checkpoints/dyn_crafter_oracle train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}


## 5. Label Fraction Sweep (0.1% to 100%), CEM Planning, and VoE Physical Scorer


In [ ]:
!python scripts/sweep_labels.py \
    --use-state-context \
    --data-root data/crafter \
    --lam-ckpt checkpoints/lam_crafter/lam_final.pt \
    --dyn-ckpt checkpoints/dyn_crafter_lam/dyn_final.pt \
    --oracle-ckpt checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --fractions {SWEEP_FRACTIONS} \
    --num-episodes {SWEEP_EPISODES} \
    --seeds {SWEEP_SEEDS} \
    --cem-horizon 4 \n    --cem-cost-metric l1 \n    --out-dir outputs/sweep_results

!python scripts/eval_voe.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --lam-checkpoint checkpoints/lam_crafter/lam_final.pt \
    --out outputs/voe_metrics.json

!python scripts/plot_paper_figures.py --metrics-json outputs/sweep_results/sweep_metrics.json --out-dir outputs/figures


### 5b. Systematic Ablation Study: Cost Metric and Horizon
Compare CEM planning performance across cost metrics (spatial_conv, l1, cosine, pooled) and horizons (H in {2, 3, 4, 6}).

In [ ]:
# Run systematic planning ablations on the Oracle dynamics model
!python scripts/eval_ablations.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --num-episodes 10 \
    --out-dir outputs/ablations_oracle

# Run systematic planning ablations on our Latent-Play dynamics model
!python scripts/eval_ablations.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --alignment-checkpoint outputs/sweep_results/alignment_f0.01_s0.pt \
    --num-episodes 10 \
    --out-dir outputs/ablations_lam


In [ ]:
!python scripts/eval_planning.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --align-ckpt outputs/sweep_results/alignment_f0.01_s0.pt \
    --target-dist 4 \
    --num-episodes {SWEEP_EPISODES}

from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")


## 6. Package Results and Logs for Download


In [ ]:
!zip -r latent_play_results.zip outputs/ checkpoints/*/log.jsonl checkpoints/*/confusion.json checkpoints/*/config.json
from google.colab import files
files.download("latent_play_results.zip")